In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [2]:
# the way we can make our model better is by training
# this is how overall machine learning goes by
# instead of procedural step by step instructions using if and so on
# u instead go on by 
# feeding the model some data and letting it predict the label 
# having a cost function to let the model know how wrong his predictions/answers to the exam were
# think of the cost function as the teacher that corrects the model's answers and gives back the grade
# then accordingly to the loss function u calculate the gradients
# gradients are a fancy way of saying "partial differentiation"
# here is where calculus comes in 
# differentiating with respect to any curve at some point gives the tangent of the curve at that point
# and a tangent is kind of a fancy way of saying "the line equation at that point"
# so after substituting with any number post getting the differentation can get u any point on that line
# that is the way of thinking abt it

# since the goal is to make the loss as small as possible 
# u would make gradients of loss with respect to each single parameter/weight
# that would give u tangent at that weight right?
# then u would just use the "GRADIENT DESCENT" equation
# and the word "descent" is named like that because u literally just descend go down on the tangent step by step till
# u get to the lowest point of loss or at least a considerable proper loss where the model can perform well
# a step is just going down or up in that curve a step down or up in that curve in terms of weights
# using the gradient descent equation wnew -= n * gradient

# n is just the learning rate a hyperparameter a multiplication of a gradient to control the size of the step itself
# make it too big and it can jump around the lowest loss point
# make it too small and it would take forever to reach a good result
# smth about it too is that normal gradient descent is called stochastic gradient descent 
# nowadays adam optimizers are made to calculate aggregations of all the weights and adapt the learning rate accordingly 
# we will talk abt that later

In [3]:
# back in the day it was firstly only a single perceptron neural network
# that was able to learn and/or and predict them correctly
# however when it came to non linear complex functions like xor 
# a single layer isnt enough
# there is a law called universal approximation law
# where it states that u can technically get the same result of a multilayer network in just a single large layer
# however there are many reasons why people dont do so
# one of the main reasons is : it is not interpretable for example to do so like 
# for example if u are trying to teach a model what each pic is about
# the first layers can see edges then second set of layers can learn curves then third set learns shapes and so on
# for a single layer that would be like trying to teach a student a whole curriculum in one sit
# further more multilayer helped back in the day in lots of things for example in the model
# 'RNNs' Recurrent Neural Networks
# we had attention for the rnns was called "bahdanau attention" where u would make it focus its attention
# on the most important hidden layers aka weighting them in terms of importances
# imagine how that would be impossible on a single layer neural network

In [4]:
# the reason people didnt want to go from single to multilayer was because they didnt know how to train a multilayer
# till a scientist published the research paper that introduced backpropagation
# a way to calculate gradients and flow them through the multilayered network
# the first impression to his discovery wasnt that much people didnt expect it to work in every situation
# however as time went it aged like wine where it proved that it can train any network 
# with some caviats though that was not his mistake the problem came from the activation function
# but again we will cover that later
# the idea of it is the "chain rule" in calculus
# where u can calculate gradients with respect of each single step then * by the next as u go on
# the idea of it came from the already existing law of chain rule in calculus where it states that
# for example dloss/dy * dy/dx == dloss/dx

In [5]:
# for now lets start training the model
from src.config import config
from src.utils import *
from src.dataloaderv1 import create_dataloaderV1    # we use the new dataloaderv1
from src.gpt import GPT8TModel
import torch
import torch.nn as nn
import tiktoken
from pathlib import Path
torch.manual_seed(42)

In [6]:
# lets now load the text, create dataloader, initialize the model
# initialize the optimizer, initialize the loss fn and load the configs
raw_text = load_text_data()

In [7]:
# now split the dataset
train_test_ratio = 0.7
train_val_ratio = 0.9

train_test_idx = int(train_test_ratio * len(raw_text))
train_val_data = raw_text[:train_test_idx]
test_data = raw_text[train_test_idx:]

# now split train to val and train
train_val_idx = int(train_val_ratio *  len(train_val_data))
train_data = train_val_data[:train_val_idx]
val_data = train_val_data[train_val_idx:]

In [8]:
print(f'train data size: {len(train_data):,} \
\nvalidation data size: {len(val_data):,} \
\ntest data size: {len(test_data):,}')

train data size: 276,240,346 
validation data size: 30,693,372 
test data size: 131,543,022


In [9]:
# now we create all 3 loaders test, train, val
tokenizer = tiktoken.get_encoding('gpt2')

train_loader = create_dataloaderV1(
    train_data,
    max_window_length=config['context_length'],
    tokenizer=tokenizer,
    stride=config['context_length'],     # make stride == max window size to decrease the chances of overfitting
    batch_size=4,
    shuffle=True,       # we care about shuffling in training to make sure the data is not ordered or smth 
                        #which can cause poor performance later on
    drop_last=True,      # since data is alot we dont care abt dropping last + we want all rows to have same exact n of tokens
    num_workers=2,
    pin_memory=True
)

val_loader = create_dataloaderV1(
    val_data,
    max_window_length=config['context_length'],
    tokenizer=tokenizer,
    stride=config['context_length'],
    batch_size=4,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True
)

test_loader = create_dataloaderV1(
    test_data,
    max_window_length=config['context_length'],
    tokenizer=tokenizer,
    stride=config['context_length'],
    batch_size=4,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=False
)

In [10]:
criterion = nn.CrossEntropyLoss()
perplexity = lambda loss: torch.exp(loss)
epochs = 4      # one epoch is one full pass of the model through the entire dataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

checkpoint_path = Path().cwd().parent / 'models' / 'V0' / 'checkpoints' 
checkpoint_path.mkdir(parents=True, exist_ok=True)

print(f"Using device: {device}")

Using device: cuda


In [11]:
# up next we initialize everything we will need
model = GPT8TModel(
    config['emb_dim'],
    config['vocab_size'],
    config['context_length'],
    config['n_layers'],
    config['n_heads'],
    config['emb_dropout'],
    config['mha_dropout'],
    config['dropout'],
    expanding_factor=6      # determines the factor at which the fnn will represent project them context vectors at higher dimensions
)

model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4, weight_decay=0.1)

In [22]:
# lets make a function that counts number of tokens in each dataset to show u why we dont need high epochs to train the model
def count_tokens(text_data, tokenizer):
    tokens = tokenizer.encode(text_data, allowed_special = {"<|endoftext|>"})
    return len(tokens)

In [23]:
# now we print out the size of the model and the number of tokens in each dataset
print_model_size(model)

The model has 59,467,264 total parameters
The model is 453.70MB
Approximately 0.44GB


In [24]:
# as u can see this architecture has the weight tying mechanism where the embedding layer and the output layer 
# share the same weights to reduce the number of parameters and regularize the model
# this model is only a 60M parameter model
# lets now see the number of tokens in each dataset
train_tokens = count_tokens(train_data, tokenizer)

In [25]:
print(f"Number of tokens in training dataset: {train_tokens:,}")

Number of tokens in training dataset: 63,209,267


In [26]:
# AS U CAN SEE THE NUMBER OF TOKENS IS 280M TOKENS APPROXIMATELY THE SAME AS THE NUMBER OF CHARACTERS IN THE DATASET
# so number is 276,697,543
# so in that case u are approximately
print(f"You are approximately training each parameter on \
{train_tokens / sum(p.numel() for p in model.parameters()):,.2f} \
tokens per parameter")

You are approximately training each parameter on 1.06 tokens per parameter


In [27]:
# an important note is that usually 
# a good spot would be training 20 tokens/parameter for a small model and 10 tokens/parameter for a large model
# however the main thing about this is for it to be "UNIQUE TOKENS" not the same tokens repeated over and over again
# however since for this dataset it is a small model and the dataset is not that large we can get away with it
# and instead we will just train it for 4 epochs
# that would make it around 4 * 1.1 = 4.4 tokens/parameter which is not a good spot but we will take this as an experiment

In [18]:
# lets define 3 functions 
# one that calculates loss and perplexity across a single batch
# and another that calculates loss and perplexity across multiple batches
# and another that evaluates the model uses the calc_loss_loader to return the loss across the train and val sets

def calc_loss_batch(model, input_batch, target_batch, device, criterion):
    """
    Calculates the loss and the perplexity for a single batch of input and target data.

    Args:
        model (nn.Module): A neural network model
        input_batch : an input batch from the dataloader
        target_batch : the target batch from the dataloader
        device : the device that the model is currently on (CPU or GPU)
        criterion : the cost function used to calculate the loss

    Returns:
        loss (float32) : the loss value for that batch
        perplexity_value (float32) : the perplexity value for that batch (exponential of the loss)
    """
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    logits = model(input_batch)
    loss = criterion(logits.flatten(0, 1), target_batch.flatten())  # we flatten the logits (0, 1) to combine batch and sequence dims

    # no need to build a computational graph for the perplexity
    with torch.no_grad():
        perplexity_value = perplexity(loss)

    return loss, perplexity_value.item()

# num_batches gives the freedom to calculate loss and perplexity across a subset of the dataset
# instead of the whole dataset
# since the dataset is large and calculating loss across the whole dataset can take a long time
def calc_loss_loader(model, data_loader, device, criterion, num_batches):
    """
    Calculates the average loss and perplexity for a given data loader.
    
    Args:
        model (nn.Module): A neural network model
        data_loader (DataLoader): A PyTorch DataLoader object containing the dataset
        device : the device that the model is currently on (CPU or GPU)
        criterion : the cost function used to calculate the loss
        num_batches (int): The number of batches to evaluate. If -1 or None, evaluates over the entire dataset.

    Returns:
        avg_loss (float32) : the average loss value across the evaluated batches
        avg_perplexity (float32) : the average perplexity value across the evaluated batches
    """
    total_loss, total_perplexity = 0.0, 0.0

    # handle the case at which the data_loader has no data smh
    if len(data_loader) == 0 or num_batches == 0:
        return float('nan'), float('nan')

    if num_batches is None or num_batches == -1:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))        # min between number of batches and length dataset (to avoid num batches being > len dataset)

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        loss, perplexity_value = calc_loss_batch(model, input_batch, target_batch, device, criterion)
        total_loss += loss.item()
        total_perplexity += perplexity_value
    
    return total_loss / num_batches, total_perplexity / num_batches     # return average loss and perplexity across the batches

# we can now define a function that evaluates the model on both the training and validation datasets
def evaluate_model(model, train_loader, val_loader, device, criterion, num_batches):
    model.eval()
    with torch.no_grad():
        train_loss, train_perplexity = calc_loss_loader(model, train_loader, device, criterion, num_batches)
        val_loss, val_perplexity = calc_loss_loader(model, val_loader, device, criterion, num_batches)

    model.train()   
    return train_loss, train_perplexity, val_loss, val_perplexity

def generate_and_print_sample(model, tokenizer, device, start_context, max_new_tokens):
    tokenIDs = text_to_tokenIDs(start_context, tokenizer)
    tokenIDs = tokenIDs.to(device)

    model.eval()
    generated_tokIDs = generate_text_tokIDs(model, tokenIDs, max_new_tokens, config['context_length'], tokenizer)
    generated_text = tokenIDs_to_text(generated_tokIDs, tokenizer)
    print(f"Generated text: {generated_text}")
    model.train()

In [19]:
def save_checkpoint(model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen, checkpoint_path,
                    best_val_loss, name):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'global_step': global_step,
        'tokens_seen': tokens_seen,
        'best_val_loss': best_val_loss,
        'train_losses': train_losses,
        'train_ppls': train_ppls,
        'val_losses': val_losses,
        'val_ppls': val_ppls,
        'all_tokens_seen': all_tokens_seen
    }

    torch.save(checkpoint, checkpoint_path / f'{name}.pt')
    # also save the best model based on validation loss

In [ ]:
def train_model(model, optimizer, criterion, train_loader, val_loader, epochs, 
                start_context, num_batches_to_eval, eval_freq, max_new_tokens, device):
    """
    Trains the given model using the given optimizer and criterion for the given number of epochs.
    Evaluates the model after each epoch and prints a sample text generated by the model. 

    Args:
        model (nn.Module): A neural network model
        optimizer (torch.optim.Optimizer): The optimizer used for training
        criterion : the cost function used to calculate the loss
        train_loader (DataLoader): A PyTorch DataLoader object containing the training dataset
        val_loader (DataLoader): A PyTorch DataLoader object containing the validation dataset
        epochs (int): The number of epochs to train the model for
        start_context (str): The initial context to generate text from
        num_batches_to_eval (int): The number of batches to evaluate the model on after each epoch
        eval_freq (int): The frequency of evaluation (in global steps)
        max_new_tokens (int): The maximum number of new tokens to generate
        device (torch.device): The device to run the model on
    
    Returns: 
        train_loss (list): A list of training loss values for each epoch
        train_perplexity (list): A list of training perplexity values for each epoch
        val_loss (list): A list of validation loss values for each epoch
        val_perplexity (list): A list of validation perplexity values for each epoch
        tokens_seen (list): A list of the number of tokens seen by the model after each epoch
    """
    train_losses, val_losses, train_ppls, val_ppls, all_tokens_seen = [], [], [], [], []
    tokens_seen, global_step = 0, -1
    best_val_loss = float('inf')        # initialize this to keep track of the best validation loss seen so far
    for epoch in range(epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            loss, perplexity = calc_loss_batch(model, input_batch, target_batch, device, criterion)

            loss.backward()

            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1    # an optimizer step

            if (global_step % eval_freq == 0):
                train_loss, train_ppl, val_loss, val_ppl = evaluate_model(model, train_loader, val_loader, device,
                                                                            criterion, num_batches_to_eval)

                train_losses.append(train_loss)
                train_ppls.append(train_ppl)
                val_losses.append(val_loss)
                val_ppls.append(val_ppl)
                all_tokens_seen.append(tokens_seen)

                print(f"Epoch [{epoch+1}/{epochs}], Step [{global_step}], Tokens Seen: {tokens_seen:,},\n\
                      Train Loss: {train_loss:.4f}, Train Perplexity: {train_ppl:.4f},\n\
                      Val Loss: {val_loss:.4f}, Val Perplexity: {val_ppl:.4f}")

                # moreover since the training will take a long time 
                # Save a checkpoint after each evaluation so training can be resumed
                # if the run is interrupted.
                save_checkpoint(model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen,
                                checkpoint_path, best_val_loss, name='latest_model')
                
                # if the best_val_loss is updated make sure to also update the best model checkpoint
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    save_checkpoint(model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen,
                                    checkpoint_path, best_val_loss, name='best_model')

                

        generate_and_print_sample(model, tokenizer, device, start_context, max_new_tokens)

    return train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen

In [21]:
# now we can finally train the model
start_context = "I was flying up in the sky when"

# THIS WILL TAKE A LONG TIME TO RUN SO MAKE SURE U HAVE A GOOD GPU AND ENOUGH VRAM
# or u can just lower the config
# or if u really want to u can train it on google colab for free and it has a good gpu and enough vram
train_model(model, optimizer, criterion, train_loader, val_loader, epochs,
            start_context, num_batches_to_eval=10, eval_freq=100, max_new_tokens=50, device=device)

Epoch [1/4], Step [0], Tokens Seen: 1,024,
                      Train Loss: 253.2802, Train Perplexity: inf,
                      Val Loss: 251.6870, Val Perplexity: inf
Epoch [1/4], Step [100], Tokens Seen: 103,424,
                      Train Loss: 30.4738, Train Perplexity: 19420011770675.1992,
                      Val Loss: 30.4814, Val Perplexity: 30805578822451.1992
Epoch [1/4], Step [200], Tokens Seen: 205,824,
                      Train Loss: 21.1095, Train Perplexity: 2163715059.2000,
                      Val Loss: 21.7947, Val Perplexity: 4961447572.8000
Epoch [1/4], Step [300], Tokens Seen: 308,224,
                      Train Loss: 16.7295, Train Perplexity: 27391277.7750,
                      Val Loss: 17.4696, Val Perplexity: 59714651.1500
Epoch [1/4], Step [400], Tokens Seen: 410,624,
                      Train Loss: 14.1638, Train Perplexity: 2021268.7875,
                      Val Loss: 14.8869, Val Perplexity: 4291188.5812
Epoch [1/4], Step [500], Tokens Seen: 

In [ ]:
# okay so as u an see we trained the model for 2 epochs for now
# saving the model and optimizer state dicts so we can resume training later on
# things to consider
# the gpu peaked at 81 degrees celsius and the vram usage peaked at 2.7GB which is not alot at all considering we have a max of 8
# another thing is that we calculated the number of tokens incorrectly in the previous cell and it was around 66M not 276,697,543 tokens
# that puts us at a disadvantage because we are training each parameter on 1.1 tokens/parameter which is not enough to train a model properly
# multiplied by 4 epochs that would make it around 4.4 tokens/parameter which is still not enough to train a model properly
# moreover it took around 500 + mins to run the training for 2 epochs

# we can take this as an experiment to see how the model performs with such a small number of tokens per parameter

In [ ]:
# now if we wanted to resume training we can just load the latest checkpoint and continue training from there
# for that lets create a function that loads the latest checkpoint and returns the model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen, best_val_loss

def load_checkpoint(checkpoint_path, model, optimizer, device):
    """
    Loads the latest checkpoint from the given path and restores the model and optimizer states.

    Args:
        checkpoint_path (Path): The path to the checkpoint file
        model (nn.Module): A neural network model
        optimizer (torch.optim.Optimizer): The optimizer used for training
    Returns:
        model (nn.Module): The model with restored state
        optimizer (torch.optim.Optimizer): The optimizer with restored state
        epoch (int): The epoch at which the checkpoint was saved
        global_step (int): The global step at which the checkpoint was saved
        tokens_seen (int): The number of tokens seen at the time of saving the checkpoint
        train_losses (list): A list of training loss values for each epoch
        train_ppls (list): A list of training perplexity values for each epoch
        val_losses (list): A list of validation loss values for each epoch
        val_ppls (list): A list of validation perplexity values for each epoch
        all_tokens_seen (list): A list of the number of tokens seen by the model after each epoch
        best_val_loss (float32): The best validation loss seen so far
    """
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    global_step = checkpoint['global_step']
    tokens_seen = checkpoint['tokens_seen']
    train_losses = checkpoint['train_losses']
    train_ppls = checkpoint['train_ppls']
    val_losses = checkpoint['val_losses']
    val_ppls = checkpoint['val_ppls']
    all_tokens_seen = checkpoint['all_tokens_seen']
    best_val_loss = checkpoint['best_val_loss']

    return model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen, best_val_loss

In [ ]:
import tqdm 

In [ ]:
# now we make a more robust training loop that can resume training from the latest checkpoint if it exists

def train_modelV1(model, optimizer, criterion, train_loader, val_loader, epochs, 
                start_context, num_batches_to_eval, eval_freq, max_new_tokens, device,
                checkpoint_path):           # the only extra argument is the checkpoint_path to check if there is a latest checkpoint
    """
    Trains the given model using the given optimizer and criterion for the given number of epochs.
    Evaluates the model after each epoch and prints a sample text generated by the model. 

    Args:
        model (nn.Module): A neural network model
        optimizer (torch.optim.Optimizer): The optimizer used for training
        criterion : the cost function used to calculate the loss
        train_loader (DataLoader): A PyTorch DataLoader object containing the training dataset
        val_loader (DataLoader): A PyTorch DataLoader object containing the validation dataset
        epochs (int): The number of epochs to train the model for
        start_context (str): The initial context to generate text from
        num_batches_to_eval (int): The number of batches to evaluate the model on after each epoch
        eval_freq (int): The frequency of evaluation (in global steps)
        max_new_tokens (int): The maximum number of new tokens to generate
        device (torch.device): The device to run the model on
        checkpoint_path (Path): The path to the checkpoint directory to save and load checkpoints from
    
    Returns: 
        train_loss (list): A list of training loss values for each epoch
        train_perplexity (list): A list of training perplexity values for each epoch
        val_loss (list): A list of validation loss values for each epoch
        val_perplexity (list): A list of validation perplexity values for each epoch
        tokens_seen (list): A list of the number of tokens seen by the model after each epoch
    """
    # first check if there is a latest checkpoint and if so load it and resume training from there
    latest_checkpoint_path = checkpoint_path / 'latest_model.pt'

    # first initialize all the variables to be used in the training loop
    train_losses, val_losses, train_ppls, val_ppls, all_tokens_seen = [], [], [], [], []
    tokens_seen, global_step, epoch = 0, -1, -1
    best_val_loss = float('inf')        # initialize this to keep track of the best validation loss seen so far

    # now we check if the latest checkpoint exists and if so load it and resume training from there
    if latest_checkpoint_path.exists():
        print(f"Found latest checkpoint at {latest_checkpoint_path}, loading from the checkpoint...")

        model, optimizer, epoch, global_step, tokens_seen,\
        train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen,\
        best_val_loss = load_checkpoint(latest_checkpoint_path, model, optimizer, device)

        print(f"Resuming training from checkpoint at epoch {epoch+1},\
               global step {global_step}, tokens seen {tokens_seen:,}, best validation loss {best_val_loss:.4f}")
    else:
        print(f"No latest checkpoint found at {latest_checkpoint_path}, starting training from scratch...")

    for epoch in range(epoch, epochs):      
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            loss, _ = calc_loss_batch(model, input_batch, target_batch, device, criterion)

            loss.backward()

            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1    # an optimizer step

            if (global_step % eval_freq == 0):
                train_loss, train_ppl, val_loss, val_ppl = evaluate_model(model, train_loader, val_loader, device,
                                                                            criterion, num_batches_to_eval)

                train_losses.append(train_loss)
                train_ppls.append(train_ppl)
                val_losses.append(val_loss)
                val_ppls.append(val_ppl)
                all_tokens_seen.append(tokens_seen)

                print(f"Epoch [{epoch+1}/{epochs}], Step [{global_step}], Tokens Seen: {tokens_seen:,},\n\
                      Train Loss: {train_loss:.4f}, Train Perplexity: {train_ppl:.4f},\n\
                      Val Loss: {val_loss:.4f}, Val Perplexity: {val_ppl:.4f}")

                # moreover since the training will take a long time 
                # Save a checkpoint after each evaluation so training can be resumed
                # if the run is interrupted.
                # if the best_val_loss is updated make sure to also update the best model checkpoint
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    save_checkpoint(model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen,
                                    checkpoint_path, best_val_loss, name='best_model')
                
                save_checkpoint(model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen,
                                checkpoint_path, best_val_loss, name='latest_model')
                

                

        generate_and_print_sample(model, tokenizer, device, start_context, max_new_tokens)


        # save yet another one at the very end 
        save_checkpoint(model, optimizer, epoch, global_step, tokens_seen, train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen,
                checkpoint_path, best_val_loss, name='latest_model')

    return train_losses, train_ppls, val_losses, val_ppls, all_tokens_seen